# 🔧 Project 3.1 — Sensor Data Cleaner
**DSA for Mechatronics | Week 03 Project**

---

> **Submission:** Upload your completed `.ipynb` to BlackBoard before the deadline.  
> **Presentation:** You will run and explain your code live in class the following session.  
> **AI tools:** Allowed. You must understand and be able to explain every line you submit.

---

## 🎯 What you are building

A motor controller records temperature every second. The raw data contains **spike outliers** — physically impossible values caused by electrical noise that appear at random moments (e.g. the motor is at 22 °C but the sensor suddenly reads 130 °C).

You will build a **4-step data cleaning pipeline**:

```
Raw signal  →  Detect spikes  →  Replace spikes  →  Find stable segment  →  Report
```

By the end you will have a function that takes messy sensor data and produces a clean, analysed output — exactly the kind of tool you would write in a real embedded systems or data acquisition job.

---

## 📐 Week 03 concepts you will practise

| Concept | Where used |
|---|---|
| **Sliding window** | Computing local average around each reading |
| **Two pointers** | Finding the longest stable segment |
| **List slicing** | Extracting sub-arrays |
| **List comprehension** | Building cleaned signal |
| **String f-strings** | Formatting the final report |

---

## ⚙️ Step 0 — Generate the sensor data

Run this cell first. It creates the `raw_signal` list you will work with throughout the project.  
Do **not** modify this cell — it uses a fixed random seed so everyone gets the same data.


In [ ]:
import random
import math

random.seed(42)  # Fixed seed — everyone gets identical data

# 200 seconds of temperature readings
# Normal behaviour: ~22 °C with small Gaussian noise (std = 0.4)
# Spike behaviour: 5% chance of reading jumping to 80–150 °C (electrical noise)

raw_signal = [
    round(22 + random.gauss(0, 0.4), 2)   if random.random() > 0.05
    else round(random.uniform(80, 150), 2)
    for _ in range(200)
]

# Quick look at what we're dealing with
print(f"Signal length : {len(raw_signal)} readings")
print(f"Min value     : {min(raw_signal):.2f} °C")
print(f"Max value     : {max(raw_signal):.2f} °C")
print(f"Values > 50°C : {sum(1 for x in raw_signal if x > 50)}  ← these are the spikes")
print()
print("First 20 readings:")
for i, v in enumerate(raw_signal[:20]):
    marker = " ← SPIKE!" if v > 50 else ""
    print(f"  t={i:03d}: {v:7.2f} °C{marker}")


---

## 🔬 Background — How the algorithms work

Before writing code, read this section carefully. It explains the logic behind each exercise.

### Sliding Window (used in Exercises 1 & 2)

A **sliding window** is a fixed-size sub-array that moves one step at a time across the data.

```
Signal:  [22.1, 22.3, 130.5, 22.4, 22.2, 22.0, 22.5]
                       ^
               We are checking this value.
               Window of neighbours (size=5):
               Left neighbours:  [22.1, 22.3]
               Right neighbours: [22.4, 22.2]
               Neighbour average = (22.1+22.3+22.4+22.2)/4 = 22.25
               Difference = |130.5 - 22.25| = 108.25  → SPIKE (threshold=20)
```

**Key idea:** instead of comparing a value to the global average (which spikes distort),
you compare it to its *local* neighbourhood. This is much more robust.

### Two Pointers (used in Exercise 3)

Two pointers is a technique where you maintain two indices, `left` and `right`, and move them
strategically to avoid nested loops.

```
Goal: find the longest segment where all values stay within ±2°C of the first value.

left=0, right=0 → expand right while condition holds
Signal: [22.1, 22.2, 21.9, 22.4, 22.8, 130.5, ...]
        L                             R         ← R stops here (130.5 breaks condition)
        Length = R - L = 5

Move left forward, reset right, repeat.
```

Without two pointers you'd use a nested loop: O(n²). With two pointers: O(n).


---

## ✏️ Exercise 1 — Detect spike indices

Write a function `detect_spikes(signal, window_size, threshold)` that returns a **list of indices** where a spike was detected.

### How to detect a spike at index `i`:
1. Collect the neighbours: readings at positions `i - window_size` to `i - 1` (left) and `i + 1` to `i + window_size` (right). Use `max(0, i-window_size)` and `min(len(signal), i+window_size+1)` to avoid going out of bounds.
2. Skip `i` itself — you're looking at neighbours only.
3. Compute the **average** of those neighbours.
4. If `abs(signal[i] - neighbour_avg) > threshold` → spike!

### Recommended parameters: `window_size=5, threshold=20`

### Expected output (with seed 42):
```
Detected 11 spikes
Indices: [3, 16, 34, 51, 68, 83, 106, 130, 152, 167, 188]
Spike values: [130.5, 98.2, ...]   (yours will match if seed=42)
```

> 💡 **Hint — collecting neighbours without including i itself:**
> ```python
> neighbours = signal[max(0, i-window_size) : i] + signal[i+1 : min(len(signal), i+window_size+1)]
> ```

> 💡 **Hint — computing average:**
> ```python
> avg = sum(neighbours) / len(neighbours)   # make sure neighbours is not empty first!
> ```


In [ ]:
def detect_spikes(signal, window_size=5, threshold=20):
    """
    Returns a list of indices where signal[i] is a spike.
    
    A spike is defined as: abs(signal[i] - local_neighbour_average) > threshold
    
    Parameters
    ----------
    signal      : list of floats  — the raw sensor readings
    window_size : int             — how many neighbours on each side to look at
    threshold   : float           — minimum deviation to count as a spike (°C)
    
    Returns
    -------
    list of int — indices of detected spikes
    """
    spike_indices = []
    
    for i in range(len(signal)):
        # --- collect left and right neighbours (excluding i itself) ---
        neighbours = signal[max(0, i - window_size) : i] + signal[i + 1 : min(len(signal), i + window_size + 1)]
        
        if len(neighbours) == 0:
            continue
        
        neighbour_avg = sum(neighbours) / len(neighbours)
        
        # YOUR CODE: check if signal[i] deviates more than threshold from neighbour_avg
        # If yes, append i to spike_indices
        
        pass  # ← replace this line
    
    return spike_indices


# ── Test your function ──────────────────────────────────────────────
spikes = detect_spikes(raw_signal)
print(f"Detected {len(spikes)} spikes")
print(f"Spike indices : {spikes}")
print(f"Spike values  : {[raw_signal[i] for i in spikes]}")

# Visual check — print signal with spikes marked
print()
print("Signal preview (first 40 readings):")
for i in range(40):
    marker = " ◄ SPIKE" if i in spikes else ""
    bar = "█" * int(max(0, min(raw_signal[i], 30)) / 1.5)
    print(f"  t={i:02d}: {raw_signal[i]:7.2f} °C  {bar}{marker}")


---

## ✏️ Exercise 2 — Clean the signal

Write `clean_signal(signal, spike_indices)` that returns a **new list** (don't modify the original)
where every spike is replaced by the average of its nearest valid neighbours.

### Rules:
- Build a `set` from `spike_indices` for fast O(1) lookup.
- For each index `i`:
  - If `i` is **not** a spike → keep `signal[i]` unchanged.
  - If `i` **is** a spike → replace it with the average of `signal[i-1]` and `signal[i+1]`, but only if those neighbours exist and are not themselves spikes.
  - If both neighbours are spikes or missing → use the global mean of all non-spike values.

### Expected result:
```
Original max : 148.73 °C
Cleaned  max :  23.10 °C   ← spikes gone
```

> 💡 **Hint — building the cleaned list with a loop:**
> ```python
> cleaned = []
> spike_set = set(spike_indices)
> global_mean = sum(v for i,v in enumerate(signal) if i not in spike_set) / (len(signal) - len(spike_set))
> 
> for i in range(len(signal)):
>     if i not in spike_set:
>         cleaned.append(signal[i])
>     else:
>         # find valid left neighbour
>         # find valid right neighbour
>         # take average, or fall back to global_mean
>         pass
> ```

> 💡 **Alternatively:** a list comprehension won't work easily here because you need to look at neighbours.
> A plain `for` loop is the right tool.


In [ ]:
def clean_signal(signal, spike_indices):
    """
    Returns a new list where every spike is replaced by the average of
    its nearest non-spike neighbours.
    Falls back to global mean of non-spike values if no valid neighbour found.
    
    Parameters
    ----------
    signal        : list of floats — original raw signal
    spike_indices : list of int    — indices identified as spikes
    
    Returns
    -------
    list of floats — cleaned signal, same length as signal
    """
    spike_set = set(spike_indices)
    
    # Compute global mean of non-spike values (used as fallback)
    non_spike_values = [signal[i] for i in range(len(signal)) if i not in spike_set]
    global_mean = sum(non_spike_values) / len(non_spike_values)
    
    cleaned = []
    
    for i in range(len(signal)):
        if i not in spike_set:
            cleaned.append(signal[i])
        else:
            # Try to find left neighbour (walk left until non-spike found)
            left_val = None
            j = i - 1
            while j >= 0 and j in spike_set:
                j -= 1
            if j >= 0:
                left_val = signal[j]
            
            # Try to find right neighbour (walk right until non-spike found)
            right_val = None
            # YOUR CODE: walk right from i+1, same idea as above
            
            # Compute replacement value
            if left_val is not None and right_val is not None:
                replacement = (left_val + right_val) / 2
            elif left_val is not None:
                replacement = left_val
            elif right_val is not None:
                replacement = right_val
            else:
                replacement = global_mean  # fallback
            
            cleaned.append(round(replacement, 2))
    
    return cleaned


# ── Test ───────────────────────────────────────────────────────────
spikes  = detect_spikes(raw_signal)
cleaned = clean_signal(raw_signal, spikes)

print(f"Original — min: {min(raw_signal):.2f}  max: {max(raw_signal):.2f}")
print(f"Cleaned  — min: {min(cleaned):.2f}  max: {max(cleaned):.2f}")
print(f"Length unchanged: {len(raw_signal)} → {len(cleaned)}")

# Show a few replaced values
print()
print("Replaced spike values:")
for i in spikes[:5]:
    print(f"  t={i:03d}: {raw_signal[i]:.2f} → {cleaned[i]:.2f}")


---

## ✏️ Exercise 3 — Find the longest stable segment (Two Pointers)

Write `longest_stable_segment(signal, tolerance)` that finds the longest contiguous segment
where every value stays within `±tolerance` °C of the **first value in the segment**.

### Algorithm (Two Pointers):

```
left = 0
best = (0, 0)       # (start_index, length)

while left < len(signal):
    right = left
    while right < len(signal) and abs(signal[right] - signal[left]) <= tolerance:
        right += 1
    # segment is signal[left : right]
    length = right - left
    if length > best[1]:
        best = (left, length)
    left = right    # jump left to end of current segment and try again
```

### Expected result (with tolerance=2.0 on cleaned signal):
```
Longest stable segment: t=001 → t=040  (40 readings)
```
(exact values depend on your cleaned signal)

> 💡 **Why jump `left = right` at the end?**  
> Once the segment breaks at `right`, we know everything from `left` to `right-1` was stable.
> Starting `left` at `right-1` would overlap and we'd recheck values we already know are stable.
> Jumping to `right` is safe because starting mid-segment can only give a shorter result.

> ⚠️ **Use the cleaned signal here, not raw_signal** — spikes would break every segment.


In [ ]:
def longest_stable_segment(signal, tolerance=2.0):
    """
    Finds the longest contiguous segment where every value is within
    ±tolerance of signal[left] (the first value of the segment).
    
    Returns
    -------
    tuple: (start_index, end_index, length)
           end_index is inclusive (last index of the segment)
    """
    best_start  = 0
    best_length = 1
    left = 0
    
    while left < len(signal):
        right = left
        
        # Expand right while condition holds
        while right < len(signal) and abs(signal[right] - signal[left]) <= tolerance:
            right += 1
        
        # Segment is signal[left : right], length = right - left
        length = right - left
        
        # YOUR CODE: update best_start and best_length if this segment is longer
        
        left = right  # jump to next unexplored position
    
    return (best_start, best_start + best_length - 1, best_length)


# ── Test ───────────────────────────────────────────────────────────
spikes  = detect_spikes(raw_signal)
cleaned = clean_signal(raw_signal, spikes)

start, end, length = longest_stable_segment(cleaned, tolerance=2.0)

print(f"Longest stable segment: t={start:03d} → t={end:03d}  ({length} readings)")
print(f"Values in segment:  min={min(cleaned[start:end+1]):.2f}  max={max(cleaned[start:end+1]):.2f}  °C")

# Also try tolerance=1.0 and tolerance=3.0
for tol in [1.0, 2.0, 3.0]:
    s, e, l = longest_stable_segment(cleaned, tolerance=tol)
    print(f"  tolerance={tol:.1f} → segment length {l}  (t={s:03d}–{e:03d})")


---

## ✏️ Exercise 4 — Generate the engineering report

Write `generate_report(raw_signal, spike_indices, cleaned_signal)` that prints a formatted
report. Use f-strings and Python string methods.

### Target output format:
```
╔══════════════════════════════════════════════╗
║       MOTOR TEMPERATURE — SENSOR REPORT     ║
╚══════════════════════════════════════════════╝

  Monitoring period : 200 s
  Spikes detected   :  11  (5.5 % of readings)
  Spike threshold   :  20 °C above local average

  Raw signal   →  min:  21.08 °C   max: 148.73 °C
  Clean signal →  min:  21.08 °C   max:  23.10 °C

  Stable segment    : t=001 → t=040  (40 s)

  Status: ✅ NOMINAL
  (NOMINAL if clean max < 60 °C, else ⚠️  WARNING)
```

> 💡 **Hint — string padding with f-strings:**
> ```python
> label = "Spikes detected"
> value = 11
> print(f"  {label:<20}: {value:>4}")
> #  Spikes detected     :   11
> ```
> `<20` = left-align in 20 chars. `>4` = right-align in 4 chars.

> 💡 **Hint — building the box border:**
> ```python
> width = 46
> print("╔" + "═" * width + "╗")
> title = "MOTOR TEMPERATURE — SENSOR REPORT"
> print("║" + title.center(width) + "║")
> print("╚" + "═" * width + "╝")
> ```


In [ ]:
def generate_report(raw_signal, spike_indices, cleaned_signal):
    """Prints a formatted engineering report to the console."""
    
    n          = len(raw_signal)
    n_spikes   = len(spike_indices)
    pct        = n_spikes / n * 100
    
    start, end, length = longest_stable_segment(cleaned_signal, tolerance=2.0)
    
    status = "✅ NOMINAL" if max(cleaned_signal) < 60 else "⚠️  WARNING"
    
    # YOUR CODE: print the report using the format shown above
    # Hints:
    #   - Use "╔", "═", "╗", "║", "╚" for the box
    #   - .center(n) centres a string in n characters
    #   - f"{value:<20}" left-aligns in 20 chars
    #   - f"{value:>6.2f}" right-aligns a float with 2 decimal places
    
    width = 48
    print("╔" + "═" * width + "╗")
    print("║" + "MOTOR TEMPERATURE — SENSOR REPORT".center(width) + "║")
    print("╚" + "═" * width + "╝")
    print()
    # ... continue here


# ── Run the full pipeline and print report ─────────────────────────
spikes  = detect_spikes(raw_signal)
cleaned = clean_signal(raw_signal, spikes)
generate_report(raw_signal, spikes, cleaned)


---

## 🚀 Stretch Goal — Visualise the result

If you have time, add a plot comparing raw vs cleaned signal.
The spike locations should be marked with red dots.

```python
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))

spikes  = detect_spikes(raw_signal)
cleaned = clean_signal(raw_signal, spikes)

ax.plot(raw_signal, color='#ef4444', alpha=0.4, linewidth=1, label='Raw')
ax.plot(cleaned,    color='#22c55e', linewidth=1.5, label='Cleaned')

# Mark spike positions
spike_vals = [raw_signal[i] for i in spikes]
ax.scatter(spikes, spike_vals, color='red', s=50, zorder=5, label='Spikes')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Motor Temperature — Raw vs Cleaned')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
```

**Stretch challenge:** Can you add a shaded rectangle over the longest stable segment?
